## Crear y transformar columnas

### Operaciones vectorizadas

Una operacion **vectorizada** se aplica de golpe a toda la columna,
sin bucles ni recorrer fila por fila:

```python
df['body_mass_kg'] = df['body_mass_g'] / 1000
```

A la derecha hay 344 valores y un solo numero. Se escribe como si fueran
dos numeros sueltos.

No es solo aritmetica: tambien son vectorizadas las **comparaciones**
(`df['body_mass_g'] > 4000`, las mascaras de la sesion 2) y las
operaciones de texto.

### Los nulos se heredan

Cualquier operacion con un `NaN` devuelve `NaN`, sin error ni aviso:
`NaN / 1000`, `NaN + 5`, `NaN * 0`.

> **Cada columna derivada hereda los huecos de la columna de la que sale.**

### Error silencioso nº 27: asignar machaca sin preguntar

```python
df['nombre'] = ...
```

Si `nombre` **no existe**, la crea. Si **existe**, la sustituye.
Las dos cosas se escriben igual y pandas no comprueba cual es el caso.

Una errata basta:

```python
df['body_mass_g'] = df['body_mass_g'] / 1000   # falta el _kg
```

**Por que no se ve:**

| | |
|---|---|
| Error o aviso | ninguno |
| `shape` | igual — no hay columna nueva |
| Nombre de la columna | **el de siempre** — dice gramos, dentro hay kilos |
| Los numeros | creibles (un `4.05` no chirria) |

**La trampa de Jupyter:** cuando el destino y el origen son la misma columna,
ejecutar la celda dos veces divide dos veces.

- **Evitarlo:** que el destino sea una columna distinta del origen.
  `df['kg'] = df['g'] / 1000` se puede ejecutar 10 veces y da lo mismo.
- **Deshacerlo:** reiniciar el kernel y ejecutar desde la celda de carga.

> Una celda que da el mismo resultado la ejecutes las veces que la ejecutes
> es una celda segura. Si el resultado depende de cuantas veces le diste al
> play, el notebook no es reproducible.

### Corchetes vs `assign`

| | Corchetes | `assign` |
|---|---|---|
| Toca el `df` original | si | no |
| Devuelve algo | no | una tabla nueva |
| Se puede encadenar | no | si |
| La columna va | al final | al final |

```python
df['kg'] = df['g'] / 1000          # modifica df
df2 = df.assign(kg=df['g'] / 1000) # df se queda igual
```

**El fallo tipico:** `df.assign(...)` sin guardar el resultado.
La tabla se construye, se muestra en pantalla y **se tira**.
Es la causa numero uno del "he ejecutado la celda y no ha cambiado nada".

> La pregunta no es "¿se ha creado?" — siempre se crea.
> Es **"¿donde ha ido a parar?"**. Sin un `=` delante, a ningun sitio.

**Encadenar**, que es para lo que sirve de verdad:

```python
resultado = (
    df
    .assign(peso_kg=df['body_mass_g'] / 1000)
    .query('peso_kg > 4')
    .groupby('species')
    .size()
)
```

Se lee de arriba abajo como una receta, sin `df_temp`, `df_temp2`,
`df_final_bueno`.

### El patron de fondo

Copia vs modificar en el sitio **no es cosa de `assign`**: recorre
pandas entero. `drop`, `rename`, `sort_values`, `fillna` y muchos mas
devuelven una tabla nueva y dejan el original intacto.

**Ante cualquier metodo nuevo, la pregunta es siempre la misma:
¿me devuelve algo, o modifica lo que tengo?**

In [39]:
import pandas as pd
import seaborn as sns
import numpy as np

df = sns.load_dataset("penguins")

In [40]:
df['body_mass_kg'] = df['body_mass_g'] / 1000

In [41]:
print(df[['body_mass_g', 'body_mass_kg']].head())
df.shape

   body_mass_g  body_mass_kg
0       3750.0          3.75
1       3800.0          3.80
2       3250.0          3.25
3          NaN           NaN
4       3450.0          3.45


(344, 8)

In [42]:
df_kg = df.assign(body_mass_kg=df['body_mass_g'] / 1000)

In [43]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,body_mass_kg
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,3.75
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,3.80
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,3.25
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,3.45


In [44]:
df2 = df.assign(peso_libras=df['body_mass_g'] / 453.6)
df2.shape
df.shape

(344, 8)

In [45]:
df2 = df.assign(peso_libras=df['body_mass_g'] / 453.6)
df2.shape
df.shape

(344, 8)

## `cut` — de numeros a categorias

`cut` coge una columna de numeros y la convierte en grupos con nombre.

```python
df['tamano'] = pd.cut(
    df['body_mass_g'],
    bins=[0, 3600, 4500, np.inf],
    labels=['pequeño', 'mediano', 'grande']
)
```

| Argumento | Que es |
|---|---|
| `bins` | los puntos de corte, las fronteras |
| `labels` | el nombre de cada tramo |

**Una etiqueta menos que cortes:** con cuatro postes se hacen tres tramos de verja.

### Que extremo entra

Por defecto el tramo se cierra **por la derecha**: el extremo izquierdo
no entra, el derecho si. En pantalla sale escrito `(2700, 3600]`.

Un peso de 3600 es **pequeño**. Mediano empieza en cuanto se pasa de 3600.

> Ojo, es al reves que `range`: alli se excluye el final, aqui se incluye
> y lo que se excluye es el principio.

La frontera es "mayor que 3600", no "3601 o mas": un 3600.5 ya seria mediano.

### Error silencioso nº 28: valores fuera de rango

Lo que queda fuera de los cortes se convierte en `NaN`. Sin error, sin aviso.

**Y se encadena con el nº 14** (`groupby` descarta los nulos de la columna
de agrupar):

```
cut      ->  fuera de rango se vuelve NaN
groupby  ->  esas filas no pertenecen a ningun grupo, se caen
```

Resultado: la suma de los grupos da **menos que `len(df)`** y nada lo dice.
Dos comportamientos razonables que, sumados, borran filas.

**Por que se cuela en produccion:** los cortes se ponen mirando el minimo
y el maximo de hoy. Funciona perfecto. El mes que viene llegan datos con
un valor mas alto, el mismo codigo se ejecuta igual, y esos registros
se evaporan.

**Solucion:** `np.inf` y `-np.inf` en los extremos. Nada se puede salir,
ni hoy ni con datos nuevos.

**Pero no siempre se quiere cerrar:**

| Los cortes son | Extremos | Por que |
|---|---|---|
| una **definicion** (tramos de IVA, edades legales) | cerrar con `inf` | nada debe quedarse fuera |
| un **resumen exploratorio** | dejarlos abiertos | el `NaN` avisa de un outlier |

Meter un pinguino de 9.000 g en "grande" seria tapar un posible error
de tecleo.

### La comprobacion: nulos antes y despues

En `tamano` acaban **dos tipos de nulo distintos**, y son indistinguibles:

| De donde viene | Que significa |
|---|---|
| el peso ya era `NaN` | no se midio — hueco honesto |
| el peso se salio de los cortes | los cortes estan mal — fallo mio |

Mirando la columna no hay forma de saber cual es cual. Por eso **no basta
con contar: hay que comparar**.

```python
df['body_mass_g'].isna().sum()   # nulos de origen
df['tamano'].isna().sum()        # nulos despues del cut
```

- Coinciden -> los cortes estan bien, esos nulos venian de casa.
- El segundo es mayor -> **la diferencia son los que se salieron de rango.**

> Es la misma idea que las comprobaciones de reestructurar: una cantidad
> que deberia conservarse. Aqui lo que se conserva es el numero de huecos.

### `cut` devuelve `category`

No devuelve `object`, devuelve `category`. Dos consecuencias:

**Las categorias tienen orden**, el de las `labels`. Al agrupar salen
pequeño, mediano, grande — no en orden alfabetico.

**El diccionario de categorias no depende de los datos.** Una columna
`category` guarda por un lado la lista de valores posibles y por otro
un numero por fila que apunta a esa lista. La lista sigue igual aunque
ninguna fila apunte a una de sus entradas.

Por eso, al agrupar, pandas recorre **las categorias declaradas**, no las
filas que hay:

| `observed` | Que hace | Es el valor |
|---|---|---|
| `False` | saca **todas** las categorias; las vacias con un `0` | **por defecto** |
| `True` | saca solo las que aparecen en los datos | hay que escribirlo |

Como casi nunca se escribe, **lo que me voy a encontrar es el `False`**.

**Y ese 0 puede ser correcto o puede estorbar:**

| Pregunta | El 0 |
|---|---|
| ¿Cuantos pinguinos grandes hay en Torgersen? | correcto — la respuesta es ninguno |
| ¿Que tamaños de pinguino hay en Torgersen? | estorba — "grande" no esta |

**Y muerde igual que el nº 25:** ese 0 cambia el denominador.

```python
conteos.mean()   # observed=False -> (35+12+0)/3 = 15.7
                 # observed=True  -> (35+12)/2   = 23.5
```

Un parametro que ni escribi cambia el resultado un 50%.

> No hay un valor correcto de `observed`: depende de si la categoria vacia
> forma parte de lo que estoy midiendo. Lo que no vale es no haberlo decidido.

In [46]:
df['tamano'] = pd.cut(
    df['body_mass_g'],
    bins=[0, 3600, 4500, np.inf],
    labels=['pequeño', 'mediano', 'grande']
)

In [47]:
print(df['tamano'].value_counts(dropna=False))
print(df['tamano'].isna().sum())

tamano
mediano    130
grande     115
pequeño     97
NaN          2
Name: count, dtype: int64
2


In [48]:
pd.cut(
    pd.Series([2700, 3600, 4500, 7000]),
    bins=[2700, 3600, 4500, 6300],
    labels=['pequeño', 'mediano', 'grande']
)

0        NaN
1    pequeño
2    mediano
3        NaN
dtype: category
Categories (3, object): ['pequeño' < 'mediano' < 'grande']

In [49]:
print(df.groupby('tamano').size())
print(df.groupby('tamano').size().sum())
print(len(df))

tamano
pequeño     97
mediano    130
grande     115
dtype: int64
342
344


/tmp/ipykernel_4526/2047964114.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('tamano').size())
/tmp/ipykernel_4526/2047964114.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('tamano').size().sum())


In [50]:
df['body_mass_g'].isna().sum()

np.int64(2)

In [51]:
print(df['body_mass_g'].isna().sum())   # nulos de origen
df['tamano'].isna().sum()        # nulos después del cut

2


np.int64(2)

In [52]:
df['tamano'].dtype

CategoricalDtype(categories=['pequeño', 'mediano', 'grande'], ordered=True, categories_dtype=object)

In [53]:
df['tamano'].cat.categories
df.groupby('tamano', observed=True).size()

tamano
pequeño     97
mediano    130
grande     115
dtype: int64

In [54]:
print(df['species'].memory_usage(deep=True))
df['species'].astype('category').memory_usage(deep=True)

22004


772